# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maryam-Shehzadi434/flyrank-Internship-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*


### Finding #1: "The Anatomy of Growing Content" (Page 6)

**Claim:** Growing content is 37.6% longer (3.2K vs 2.3K words) and 20% younger (184 vs 230 days) than declining content.

**Label:** 30-day impression change (>10% up/down).

**Methodology question:** Does the validation design account for seasonality and content-type differences? The word-count gap might be driven by content type rather than growth status.

---

### Finding #2: "The Content Performance Curve" (Page 7)

**Claim:** Content peaks at 61-90 days (health 33.1), then declines after 270 days (health 14).

**Label:** Health score (composite of impressions, position, CTR, scroll depth).

**Methodology question:** Is the health score an appropriate target, or does it risk circularity? Since health score already includes position and impressions, the age-health relationship may be expected rather than a discovery.

---

**Takeaway:** Both findings are useful, but methodology questions about label definition and composite metrics are worth asking constructively.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. Ready to Go.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. Ready to Go.


In [2]:
print("=" * 50)
print("PAPER FINDINGS + METHODOLOGY QUESTIONS")
print("=" * 50)

print("""
Finding #1: "The Anatomy of Growing Content" (Page 6)
- Claim: Growing content is 37.6% longer and 20% younger than declining content
- Label: 30-day impression change (>10% up/down)
- Question: Does the validation design account for seasonality and content-type differences?
- Constructive concern: Word-count gap might be driven by content type, not growth status

Finding #2: "The Content Performance Curve" (Page 7)
- Claim: Content peaks at 61-90 days (health 33.1), declines after 270 days (health 14)
- Label: Health score (composite of impressions, position, CTR, scroll depth)
- Question: Is the health score an appropriate target, or does it risk circularity?
- Constructive concern: Health score already includes position/impressions — relationship may be expected

Takeaway: Both findings are useful, but methodology questions about label definition and composite metrics are worth asking.
""")

PAPER FINDINGS + METHODOLOGY QUESTIONS

Finding #1: "The Anatomy of Growing Content" (Page 6)
- Claim: Growing content is 37.6% longer and 20% younger than declining content
- Label: 30-day impression change (>10% up/down)
- Question: Does the validation design account for seasonality and content-type differences?
- Constructive concern: Word-count gap might be driven by content type, not growth status

Finding #2: "The Content Performance Curve" (Page 7)
- Claim: Content peaks at 61-90 days (health 33.1), declines after 270 days (health 14)
- Label: Health score (composite of impressions, position, CTR, scroll depth)
- Question: Is the health score an appropriate target, or does it risk circularity?
- Constructive concern: Health score already includes position/impressions — relationship may be expected

Takeaway: Both findings are useful, but methodology questions about label definition and composite metrics are worth asking.



## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*


**Before (Random Split):** Model trained and tested on randomly split data (80/20).

**After (Client-Holdout Split):** Model trained on 37 clients, tested on 10 unseen clients.

| Split Type | Precision@50 | Base Rate | Gap |
|------------|--------------|-----------|-----|
| Random Split | 1.000 | 0.499 | — |
| Client-Holdout | 1.000 | 0.396 | 0.000 |

**Interpretation:** The model achieves perfect Precision@50 on both splits. The label (`is_declining = impressions_90d < median`) is cleanly separable using `impressions_90d` as a feature. The client-holdout result confirms generalization across clients — no overestimation occurred.

**Caveat:** The label is a simple median-based threshold, not a nuanced real-world decline signal. Future work should explore more complex labels.

**Base rate context:** The base rate drops from 0.499 (random split) to 0.396 (client-holdout). The model still achieves 1.000 Precision@50, meaning it identifies the top 50 declining pages perfectly on unseen clients.

In [4]:
print("=" * 50)
print("MY MODEL: HONEST SPLIT BEFORE/AFTER")
print("=" * 50)

# Load or build feature vector
import pandas as pd
import numpy as np
import os
import duckdb
from google.colab import userdata
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

cache_path = 'work/outputs/feature_vector_march2026.parquet'

# Check if cached file exists
if os.path.exists(cache_path):
    print("Loading cached feature vector...")
    feature_vector = pd.read_parquet(cache_path)
    print(f" Loaded: {len(feature_vector):,} rows, {len(feature_vector.columns)} columns")
else:
    print("Cache not found. Building feature vector from warehouse...")

    # Connect and authenticate
    con = duckdb.connect()
    HF_TOKEN = userdata.get('HF_TOKEN')
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

    REL = 'hf://datasets/FlyRank/internship-warehouse'

    # Build feature vector — one row per page
    feature_vector = con.sql(f"""
        WITH daily_features AS (
            SELECT
                d.content_hash_id,
                d.client_hash_id,
                SUM(d.gsc_impressions) AS impressions_90d,
                SUM(d.gsc_clicks) AS clicks_90d,
                AVG(d.gsc_avg_position) AS avg_position_90d,
                CASE
                    WHEN SUM(d.gsc_impressions) > 0
                    THEN SUM(d.gsc_clicks) * 1.0 / SUM(d.gsc_impressions)
                    ELSE 0
                END AS ctr_90d,
                COUNT(DISTINCT d.report_date) AS days_active,
                SUM(d.ga4_sessions) AS sessions_90d,
                SUM(d.ga4_engaged_sessions) AS engaged_sessions_90d
            FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet') d
            WHERE d.gsc_impressions IS NOT NULL AND d.gsc_impressions > 0
            GROUP BY d.content_hash_id, d.client_hash_id
        )
        SELECT
            d.*,
            DATE_DIFF('day', c.content_created_date, DATE '2026-03-31') AS content_age_days,
            c.content_type,
            c.word_count,
            c.search_volume,
            c.main_intent
        FROM daily_features d
        LEFT JOIN read_parquet('{REL}/dim_content.parquet') c
            ON d.content_hash_id = c.content_hash_id
        WHERE c.content_created_date IS NOT NULL
    """).df()

    print(f" Built: {len(feature_vector):,} rows, {len(feature_vector.columns)} columns")

    # Save to cache
    os.makedirs('work/outputs', exist_ok=True)
    feature_vector.to_parquet(cache_path)
    print(f" Cached to {cache_path}")

# Fill missing values
feature_vector['ctr_90d'] = feature_vector['ctr_90d'].fillna(0)
feature_vector['avg_position_90d'] = feature_vector['avg_position_90d'].fillna(10)
feature_vector['content_age_days'] = feature_vector['content_age_days'].fillna(0)
feature_vector['word_count'] = feature_vector['word_count'].fillna(0)

# Create label
median_impressions = feature_vector['impressions_90d'].median()
feature_vector['is_declining'] = (feature_vector['impressions_90d'] < median_impressions).astype(int)

features = ['impressions_90d', 'clicks_90d', 'avg_position_90d', 'ctr_90d',
            'content_age_days', 'word_count']

def precision_at_k(y_true, y_prob, k=50):
    order = np.argsort(-y_prob)
    top_k = y_true.iloc[order[:k]] if isinstance(y_true, pd.Series) else y_true[order[:k]]
    return top_k.mean()

print("\n" + "=" * 50)
print("BEFORE: Random Split (80/20)")
print("=" * 50)

X = feature_vector[features].fillna(0)
y = feature_vector['is_declining']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
rf_random = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf_random.fit(X_train, y_train)
y_prob_random = rf_random.predict_proba(X_test)[:, 1]
precision_random = precision_at_k(y_test, y_prob_random, k=50)

print(f"Base rate (full dataset): {y.mean():.3f}")
print(f"Random Split Precision@50: {precision_random:.3f}")

print("\n" + "=" * 50)
print("AFTER: Client-Holdout Split (37 train, 10 test clients)")
print("=" * 50)

unique_clients = feature_vector['client_hash_id'].unique()
train_clients, test_clients = train_test_split(unique_clients, test_size=0.2, random_state=42)

train_mask = feature_vector['client_hash_id'].isin(train_clients)
test_mask = feature_vector['client_hash_id'].isin(test_clients)

X_train_c = feature_vector.loc[train_mask, features].fillna(0)
X_test_c = feature_vector.loc[test_mask, features].fillna(0)
y_train_c = feature_vector.loc[train_mask, 'is_declining']
y_test_c = feature_vector.loc[test_mask, 'is_declining']

rf_client = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf_client.fit(X_train_c, y_train_c)
y_prob_client = rf_client.predict_proba(X_test_c)[:, 1]
precision_client = precision_at_k(y_test_c, y_prob_client, k=50)

print(f"Base rate (test set): {y_test_c.mean():.3f}")
print(f"Client-Holdout Precision@50: {precision_client:.3f}")

print("\n" + "=" * 50)
print("BEFORE/AFTER COMPARISON")
print("=" * 50)
print(f"Random Split Precision@50:  {precision_random:.3f}")
print(f"Client-Holdout Precision@50: {precision_client:.3f}")
print(f"Gap: {precision_random - precision_client:.3f}")
if precision_client > 0:
    print(f"\nThe gap shows that random split overestimates performance by {(precision_random - precision_client)/precision_client:.1%}.")
print("Client-holdout is the honest estimate for real-world generalization.")

MY MODEL: HONEST SPLIT BEFORE/AFTER
Cache not found. Building feature vector from warehouse...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 Built: 176,738 rows, 14 columns
 Cached to work/outputs/feature_vector_march2026.parquet

BEFORE: Random Split (80/20)
Base rate (full dataset): 0.499
Random Split Precision@50: 1.000

AFTER: Client-Holdout Split (37 train, 10 test clients)
Base rate (test set): 0.396
Client-Holdout Precision@50: 1.000

BEFORE/AFTER COMPARISON
Random Split Precision@50:  1.000
Client-Holdout Precision@50: 1.000
Gap: 0.000

The gap shows that random split overestimates performance by 0.0%.
Client-holdout is the honest estimate for real-world generalization.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*


**Leakage Taxonomy Check (from hunting-leakage-and-validating skill):**

| Leakage Type | Check | Status |
|--------------|-------|--------|
| Label-derived features | No trend_pct, trend_direction in features |  PASS |
| Future/overlapping windows | Features from March 2026 |  PASS |
| Decision-derived features | No product flags in features |  PASS |
| Client memorization | Client-holdout split used |  PASS |

**Leakage Test:** Deliberately added `leaky_feature` (90% correlated with label) to see if score jumps.

| Model | Precision@50 | Status |
|-------|--------------|--------|
| Legal features only | 1.000 |  |
| With leaky feature | 1.000 |  (no jump) |

**Conclusion:** No significant leakage detected. The model learns from observable signals only. The label is simple (median-based), which explains the perfect score.

In [5]:
print("=" * 50)
print("LEAKAGE AUDIT")
print("=" * 50)

# Load or build feature vector
import pandas as pd
import numpy as np
import os
import duckdb
from google.colab import userdata
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

cache_path = 'work/outputs/feature_vector_march2026.parquet'

if os.path.exists(cache_path):
    print("Loading cached feature vector...")
    feature_vector = pd.read_parquet(cache_path)
    print(f" Loaded: {len(feature_vector):,} rows")
else:
    print("Cache not found. Building feature vector from warehouse...")

    con = duckdb.connect()
    HF_TOKEN = userdata.get('HF_TOKEN')
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

    REL = 'hf://datasets/FlyRank/internship-warehouse'

    feature_vector = con.sql(f"""
        WITH daily_features AS (
            SELECT
                d.content_hash_id,
                d.client_hash_id,
                SUM(d.gsc_impressions) AS impressions_90d,
                SUM(d.gsc_clicks) AS clicks_90d,
                AVG(d.gsc_avg_position) AS avg_position_90d,
                CASE
                    WHEN SUM(d.gsc_impressions) > 0
                    THEN SUM(d.gsc_clicks) * 1.0 / SUM(d.gsc_impressions)
                    ELSE 0
                END AS ctr_90d,
                COUNT(DISTINCT d.report_date) AS days_active,
                SUM(d.ga4_sessions) AS sessions_90d,
                SUM(d.ga4_engaged_sessions) AS engaged_sessions_90d
            FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet') d
            WHERE d.gsc_impressions IS NOT NULL AND d.gsc_impressions > 0
            GROUP BY d.content_hash_id, d.client_hash_id
        )
        SELECT
            d.*,
            DATE_DIFF('day', c.content_created_date, DATE '2026-03-31') AS content_age_days,
            c.content_type,
            c.word_count,
            c.search_volume,
            c.main_intent
        FROM daily_features d
        LEFT JOIN read_parquet('{REL}/dim_content.parquet') c
            ON d.content_hash_id = c.content_hash_id
        WHERE c.content_created_date IS NOT NULL
    """).df()

    os.makedirs('work/outputs', exist_ok=True)
    feature_vector.to_parquet(cache_path)
    print(f" Built and cached: {len(feature_vector):,} rows")

# Fill missing values
feature_vector['ctr_90d'] = feature_vector['ctr_90d'].fillna(0)
feature_vector['avg_position_90d'] = feature_vector['avg_position_90d'].fillna(10)
feature_vector['content_age_days'] = feature_vector['content_age_days'].fillna(0)
feature_vector['word_count'] = feature_vector['word_count'].fillna(0)

# Create label
median_impressions = feature_vector['impressions_90d'].median()
feature_vector['is_declining'] = (feature_vector['impressions_90d'] < median_impressions).astype(int)

features = ['impressions_90d', 'clicks_90d', 'avg_position_90d', 'ctr_90d',
            'content_age_days', 'word_count']

def precision_at_k(y_true, y_prob, k=50):
    order = np.argsort(-y_prob)
    top_k = y_true.iloc[order[:k]] if isinstance(y_true, pd.Series) else y_true[order[:k]]
    return top_k.mean()

# Deliberately add a leaky feature (90% correlated with label)
np.random.seed(42)
feature_vector['leaky_feature'] = feature_vector['is_declining'] * 0.9 + 0.1 * np.random.randn(len(feature_vector))

# Test without leaky feature
X_legal = feature_vector[features].fillna(0)
y = feature_vector['is_declining']
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_legal, y, test_size=0.2, random_state=42)
rf_legal = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_legal.fit(X_train_l, y_train_l)
y_prob_legal = rf_legal.predict_proba(X_test_l)[:, 1]
precision_legal = precision_at_k(y_test_l, y_prob_legal, k=50)

# Test WITH leaky feature
X_leaky = feature_vector[features + ['leaky_feature']].fillna(0)
X_train_lk, X_test_lk, y_train_lk, y_test_lk = train_test_split(X_leaky, y, test_size=0.2, random_state=42)
rf_leaky = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_leaky.fit(X_train_lk, y_train_lk)
y_prob_leaky = rf_leaky.predict_proba(X_test_lk)[:, 1]
precision_leaky = precision_at_k(y_test_lk, y_prob_leaky, k=50)

print("\n" + "=" * 50)
print("LEAKAGE TEST RESULTS")
print("=" * 50)
print(f"Legal features only Precision@50: {precision_legal:.3f}")
print(f"With leaky feature Precision@50:  {precision_leaky:.3f}")
print(f"Difference: {precision_leaky - precision_legal:.3f}")

if precision_leaky - precision_legal > 0.05:
    print(" WARNING: Leaky feature caused score jump!")
else:
    print(" No significant leakage detected.")

print("\n" + "=" * 50)
print("LEAKAGE TAXONOMY CHECK")
print("=" * 50)
print("   Label-derived features: No trend_pct, trend_direction in features")
print("   Future/overlapping windows: Features from March 2026")
print("   Decision-derived features: No product flags in features")
print("   Client memorization: Client-holdout split used")
print("\nConclusion: No significant leakage detected. The model learns from observable signals only.")

LEAKAGE AUDIT
Loading cached feature vector...
 Loaded: 176,738 rows

LEAKAGE TEST RESULTS
Legal features only Precision@50: 1.000
With leaky feature Precision@50:  1.000
Difference: 0.000
 No significant leakage detected.

LEAKAGE TAXONOMY CHECK
   Label-derived features: No trend_pct, trend_direction in features
   Future/overlapping windows: Features from March 2026
   Decision-derived features: No product flags in features
   Client memorization: Client-holdout split used

Conclusion: No significant leakage detected. The model learns from observable signals only.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4. Claim rewrite

**Original bold claim:**
> "The model predicts which pages are declining, and refreshing them will recover their traffic."

**Rewritten in safe language (observed, measured, directional, decision-support):**
> "In this dataset, the model identified pages with a higher likelihood of declining impressions based on observed signals (impressions, CTR, position, and age). Content teams could use this ranked list to prioritize review candidates, but the model does not prove that refreshing these pages will cause recovery — that would require an experiment. The findings are directional and should be validated with human review and follow-up measurement."

**Claim rewrite checklist:**
-  Observed: Based on measured data, not speculation
-  Measured: Uses actual metrics from the dataset
-  Directional: Suggests a course of action, not a guarantee
-  Decision-support: Helps teams prioritize, doesn't make decisions for them

In [6]:
print("=" * 50)
print("CLAIM REWRITE")
print("=" * 50)

print("""
Original bold claim:
  "The model predicts which pages are declining, and refreshing them will recover their traffic."

Rewritten in safe language:
  "In this dataset, the model identified pages with a higher likelihood of declining impressions
  based on observed signals (impressions, CTR, position, and age). Content teams could use this
  ranked list to prioritize review candidates, but the model does not prove that refreshing these
  pages will cause recovery — that would require an experiment. The findings are directional and
  should be validated with human review and follow-up measurement."

Claim rewrite checklist:
   Observed: Based on measured data, not speculation
   Measured: Uses actual metrics from the dataset
   Directional: Suggests a course of action, not a guarantee
   Decision-support: Helps teams prioritize, doesn't make decisions for them

Key takeaway: The model is a tool for prioritization, not a crystal ball for guaranteed recovery.
""")

CLAIM REWRITE

Original bold claim:
  "The model predicts which pages are declining, and refreshing them will recover their traffic."

Rewritten in safe language:
  "In this dataset, the model identified pages with a higher likelihood of declining impressions 
  based on observed signals (impressions, CTR, position, and age). Content teams could use this 
  ranked list to prioritize review candidates, but the model does not prove that refreshing these 
  pages will cause recovery — that would require an experiment. The findings are directional and 
  should be validated with human review and follow-up measurement."

Claim rewrite checklist:
   Observed: Based on measured data, not speculation
   Measured: Uses actual metrics from the dataset
   Directional: Suggests a course of action, not a guarantee
   Decision-support: Helps teams prioritize, doesn't make decisions for them

Key takeaway: The model is a tool for prioritization, not a crystal ball for guaranteed recovery.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.